### Learning Objectives
- Define schemas explicitly vs auto-inference
- Understand all PySpark data types
- Handle complex/nested data structures
- Best practices for schema design

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession
    .builder
    .master('local[*]')
    .appName("Module02_Schema_Design")
    .config('spark.sql.shuffle.partitions','8')
    .config('spark.driver.memory','4g')
    .config('spark.sql.repl.eagerEval.enabled','true')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print(f"Spark Version: {spark.version}")

Spark Version: 4.1.2


In [2]:
# 1.1 Schema Inference (NOT recommended for production)

cust_infer_df = spark.read.csv('data/customers.csv',header=True,inferSchema=True)

print("InferSchema : ")
cust_infer_df.printSchema()
print(f"Row Count : {cust_infer_df.count()}")

InferSchema : 
root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- is_active: boolean (nullable = true)

Row Count : 5000


In [3]:
# 1.2 Explicit Schema using StructType
# ============================================================

customer_schema = StructType([
    StructField("customer_id", IntegerType(), False),  # NOT NULL
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("is_active", BooleanType(), True)
])

cust_explicit_df = spark.read.csv('data/customers.csv',header=True,schema=customer_schema)

print("Explicit Schema")
cust_explicit_df.printSchema()
cust_explicit_df.show(5)

Explicit Schema
root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- is_active: boolean (nullable = true)

+-----------+----------+---------+--------------------+------------+--------+-----+-----------+----------------+---------+
|customer_id|first_name|last_name|               email|       phone|    city|state|signup_date|customer_segment|is_active|
+-----------+----------+---------+--------------------+------------+--------+-----+-----------+----------------+---------+
|          1|   Charles|    Brown|charles.brown1@em...|555-381-5012|New York|   NY| 2022-01-29|         Regular|     true|
|          2|     Nancy|    Brown|nancy.brown2@emai...|555-658-2424|  Austin|   T

In [4]:
# 1.3 DDL String Schema (Quick & Clean)
# ============================================================

ddl_schema = """
    customer_id INT NOT NULL,
    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    state STRING,
    signup_date DATE,
    customer_segment STRING,
    is_active BOOLEAN
"""

df_ddl = spark.read.csv("data/customers.csv",
                        header=True,
                        schema=ddl_schema)

print("DDL Schema:")
df_ddl.printSchema()

DDL Schema:
root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- is_active: boolean (nullable = true)



In [5]:
# 2.1 Demonstrating Data Types

complex_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("scores", ArrayType(IntegerType()), True),  # Array
    StructField("attributes", MapType(StringType(), StringType()), True),  # Map
    StructField("address", StructType([  # Nested Struct
        StructField("street", StringType(), True),
        StructField("city", StringType(), True),
        StructField("zip", StringType(), True)
    ]), True),
    StructField("is_active", BooleanType(), True),
    StructField("salary", DoubleType(), True),
    StructField("join_date", StringType(), True)
])

data = [
    (1, "Alice", [90, 85, 92], {"dept": "Engineering", "level": "Senior"},
     ("123 Main St", "New York", "10001"), True, 95000.0, "2023-01-15"),
    (2, "Bob", [88, 92, 85], {"dept": "Marketing", "level": "Junior"},
     ("456 Oak Ave", "Los Angeles", "90001"), True, 72000.0, "2023-03-20"),
    (3, "Charlie", [78, 82, 88], {"dept": "Sales", "level": "Mid"},
     ("789 Pine Rd", "Chicago", "60601"), False, 68000.0, "2022-11-10")
]

complex_df = spark.createDataFrame(data,complex_schema)
complex_df.printSchema()
complex_df.show(truncate=False)

root
 |-- id: integer (nullable = false)
 |-- name: string (nullable = true)
 |-- scores: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- attributes: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- address: struct (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- zip: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- salary: double (nullable = true)
 |-- join_date: string (nullable = true)

+---+-------+------------+--------------------------------------+---------------------------------+---------+-------+----------+
|id |name   |scores      |attributes                            |address                          |is_active|salary |join_date |
+---+-------+------------+--------------------------------------+---------------------------------+---------+-------+----------+
|1  |Alice  |[90, 85, 92]|{level -> Senior, dept -> E

In [6]:
# 2.2 Working with Complex Types

# Access Array elements
complex_df.select(
    "name",
    col("scores")[0].alias("first_scoree"),
    size("scores").alias("num_size"),
    array_contains("scores",90).alias("has_90")
).show(5)

+-------+------------+--------+------+
|   name|first_scoree|num_size|has_90|
+-------+------------+--------+------+
|  Alice|          90|       3|  true|
|    Bob|          88|       3| false|
|Charlie|          78|       3| false|
+-------+------------+--------+------+



In [7]:
# Access Map elements

complex_df.select(
    "name",
    col("attributes")['dept'].alias("departments"),
    col("attributes")['level'].alias('level'),
    map_keys("attributes").alias("all_keys"),
    map_values("attributes").alias("all_values")
).show()

+-------+-----------+------+-------------+--------------------+
|   name|departments| level|     all_keys|          all_values|
+-------+-----------+------+-------------+--------------------+
|  Alice|Engineering|Senior|[level, dept]|[Senior, Engineer...|
|    Bob|  Marketing|Junior|[level, dept]| [Junior, Marketing]|
|Charlie|      Sales|   Mid|[level, dept]|        [Mid, Sales]|
+-------+-----------+------+-------------+--------------------+



In [8]:
# Access Nested Struct

complex_df.select(
    "name",
    col("address").street.alias("street"),
    col("address").city.alias("City"),
    col("address").zip.alias("zipcode")
)



name,street,City,zipcode
Alice,123 Main St,New York,10001
Bob,456 Oak Ave,Los Angeles,90001
Charlie,789 Pine Rd,Chicago,60601


In [9]:
# Explode Arrays and Maps

complex_df.select("name",explode("scores").alias("score"))

name,score
Alice,90
Alice,85
Alice,92
Bob,88
Bob,92
Bob,85
Charlie,78
Charlie,82
Charlie,88


In [11]:
# Explode map - creates a row for each key-value pair

complex_df.select("name",explode("attributes").alias("key","value"))

name,key,value
Alice,level,Senior
Alice,dept,Engineering
Bob,level,Junior
Bob,dept,Marketing
Charlie,level,Mid
Charlie,dept,Sales


In [21]:
def validate_schema(df, expected_schema):
    """Validate DataFrame schema matches expected schema."""

    errors = []

    # Expected schema ko dict me convert
    expected_fields = {f.name: f.dataType for f in expected_schema.fields}

    # Check actual fields
    for field in df.schema.fields:
        if field.name not in expected_fields:
            errors.append(f"Unexpected column: {field.name}")
        elif field.dataType != expected_fields[field.name]:
            errors.append(f"Type mismatch for {field.name}: "
                          f"expected {expected_fields[field.name]}, got {field.dataType}")

    # Check missing columns
    for field_name in expected_fields:
        if field_name not in [f.name for f in df.schema.fields]:
            errors.append(f"Missing column: {field_name}")

    return errors


# Test validation

errors = validate_schema(cust_explicit_df,customer_schema)

if errors:
    print("Schema validation errors : ")
    for e in errors:
        print(f"  - {e}")
else:
    print("Schema Validation passed")        
                              

Schema Validation passed


In [ ]:
# Bonus : Handling Corrupt Records
# Create sample CSV with correct data

import tempfile
import os

csv_content = """id,name,value
1,Alice,100
2,Bob,200
3,Charli,not_a_number
4,Diana"""

csv_path = os.path.join(tempfile.gettempdir(),"test_correct.csv")

with open(csv_path,"w") as F:
    F.write(csv_content)

df_permissive = spark.read.csv(csv_path,header=True,mode='PERMISSIVE',columnNameOfCorruptRecord="_corrupt")
print("Permissive mode: ")
df_permissive.show()    



Permissive mode: 
+---+------+------------+
| id|  name|       value|
+---+------+------------+
|  1| Alice|         100|
|  2|   Bob|         200|
|  3|Charli|not_a_number|
|  4| Diana|        NULL|
+---+------+------------+



In [27]:
# DROPMALFORMED mode
df_drop = spark.read.csv(csv_path,header=True,mode="DROPMALFORMED")
print("\ndropmalformed mode: ")
df_drop.show()


dropmalformed mode: 
+---+------+------------+
| id|  name|       value|
+---+------+------------+
|  1| Alice|         100|
|  2|   Bob|         200|
|  3|Charli|not_a_number|
+---+------+------------+



In [28]:
# FAILFAST MODE

try:
    df_fail = spark.read.csv(csv_path,header=True,mode="FAILFAST")
except Exception as e:
    print(f"\nFailfast mode threw exception: {e}")    

In [31]:


os.remove(csv_path)

In [ ]:
spark.stop()
print("SparkSession stopped")